# 03 — Integration Validation: Population ↔ GeoNames Settlement Joinability

## Purpose

Validate deterministic joinability between government (KSH) settlement
population data and GeoNames settlement records, and establish a
correct, evidenced strategy for resolving settlement name collisions.

## Findings

**Source compatibility.** One non-settlement government record exists
(`KSH code = 0`, `LAKCÍM NÉLKÜLI` — "no fixed address") and is excluded
here. It is *not* dropped from the production pipeline, since
`fact_population` references it; `dim_settlement` keeps a row for it
with intentionally null coordinates.

**Normalization.** Budapest districts use Roman numerals in GeoNames
(`Budapest I.`) vs. numeric in government data. Converting to
zero-padded numeric form (`Budapest 01`) resolves this.

**Settlement name is not a unique key in Hungary.** Joining on
normalized name alone produces dozens of KSH settlements with more
than one GeoNames match, with coordinate spreads up to ~400 km within
a single "duplicate" group.

**County disambiguation requires a crosswalk, used selectively.**
GeoNames' `admin1_code` numbering for Hungarian counties doesn't match
KSH's `county_code` numbering — crosswalked by name via GeoNames'
published `admin1CodesASCII.txt`. Critically, county must only be
checked for names that are *actually ambiguous*: some individual
GeoNames records carry an incorrect or stale `admin1_code` even when
they're the sole, correct candidate for a non-colliding name.
Requiring county-match universally breaks legitimate matches for
exactly those records. The fix is a three-tier strategy:

1. **Unambiguous names** (single GeoNames candidate) match on name
   alone — sidestepping any bad `admin1_code`, since there's nothing
   to disambiguate.
2. **Ambiguous names** (multiple GeoNames candidates) match on
   `(name, county_code)`, which correctly resolves cross-county
   collisions.
3. **Residual same-county duplicates** (6 names: GeoNames carries two
   records for the same place/county) are resolved by preferring
   higher recorded `population` — a real population is strong
   evidence of being the actual settlement rather than a near-empty
   satellite hamlet — falling back to the most recently modified
   record only when population also ties. This fallback is weaker
   (modification date isn't causally tied to geographic correctness)
   and was spot-checked against one real case (`Szilvágy`) rather than
   assumed.

## Result

3178 / 3178 KSH settlements matched — 100.00% coverage.

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import re

from hpm.settings import settings

pd.set_option("display.max_colwidth", None)

## Load Governmental Settlement Population Data

Population data profiling revealed one administrative change: Balatonakarattya is a distinct settlement from 2014 on, meaning 3178 records after 2014 and 3176 before. Select a population Excel file that contains all 3178 records. Insert an aggregate Budapest record.

In [ ]:
N_GOV_RECORDS = 3178

excel_files = list(settings.raw_population.iterdir())
i = 0
gov = pd.read_excel(
    excel_files[i],
    sheet_name=0,
    header=1,
    index_col=None,
)

while len(gov) != N_GOV_RECORDS:
    i += 1
    gov = pd.read_excel(
        excel_files[i],
        sheet_name=0,
        header=1,
        index_col=None,
    )


gov = gov.rename(columns={
    "KSH\nkód": "ksh_code",
    "Település": "ksh_name",
    "Megye": "county_name",
    "Vármegye": "county_name",
    "Megye\nkód": "county_code",
    "Vármegye\nkód": "county_code"
})

EXCLUDED_KSH_CODES = {0}  # Corresponds to "LAKCÍM NÉLKÜLI"
gov = gov[~gov["ksh_code"].isin(EXCLUDED_KSH_CODES)].copy()

HCSO_CODE_BUDAPEST = 1357
budapest_county_code = gov[gov["county_name"] == "BUD"]["county_code"].unique()[0]
budapest_row = pd.DataFrame([{
    "ksh_code": HCSO_CODE_BUDAPEST,
    "ksh_name": "Budapest",
    "county_code": budapest_county_code,
    "county_name": "BUD",
    "Település\ntípusa": "főváros",
}])

gov = pd.concat([gov, budapest_row], ignore_index=True)
display(gov)

## Load GeoNames Settlements

In [ ]:
GEONAMES_COLUMNS = [
    "geonameid","name","asciiname","alternatenames",
    "latitude","longitude","feature class","feature code",
    "county code","cc2","admin1 code","admin2 code",
    "admin3 code","admin4 code","population",
    "elevation","dem","timezone","modification date"
]

geo = pd.read_csv(
    settings.raw_settlements / "HU.txt",
    sep="\t",
    header=None,
    names=GEONAMES_COLUMNS,
)

ALLOWED_FEATURE_CODES = [
    "PPL",      # Populated places
    "PPLA",     # County seat
    "PPLA2",    # "járás" seat
    "PPLC",     # Capital, i.e., Budapest
    "ADM2",     # County district "járás" + capital districts
]

mask = geo["feature code"].isin(ALLOWED_FEATURE_CODES)

geo = geo[mask].copy()

display(geo)

## Normalization

In [ ]:
ROMAN_TO_INT = {
    "I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6, "VII": 7,
    "VIII": 8, "IX": 9, "X": 10, "XI": 11, "XII": 12, "XIII": 13,
    "XIV": 14, "XV": 15, "XVI": 16, "XVII": 17, "XVIII": 18, "XIX": 19,
    "XX": 20, "XXI": 21, "XXII": 22, "XXIII": 23,
}

def normalize(name: str) -> str:
    name = name.strip()

    # Normalize whitespace
    name = re.sub(r"\s+", " ", name)

    # Convert district formats
    match = re.match(r"Budapest ([IVX]+)\.*", name)

    if match:
        roman_dist = match.group(1)
        district = ROMAN_TO_INT[roman_dist]
        return f"Budapest {district:02}"

    return name

gov["norm"] = gov["ksh_name"].apply(normalize)
geo["norm"] = geo["name"].apply(normalize)

## Coordinate Spread Helper
Used both to demonstrate the original name-only problem and to verify the county-based fix actually shrinks it.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    """Great-circle distance between two points, in kilometers."""
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


def max_pairwise_km(group: pd.DataFrame) -> float:
    coords = group[["latitude", "longitude"]].to_numpy()
    if len(coords) < 2:
        return 0.0
    return max(
        haversine_km(*coords[i], *coords[j])
        for i in range(len(coords))
        for j in range(i + 1, len(coords))
    )

## Finding: Name Alone Produces Duplicates

Joining on normalized name alone, without any county disambiguation.

In [ ]:
probe = gov.merge(geo, on="norm", how="left")
name_only_dupes = probe.groupby("norm").size()
name_only_dupe_groups = probe[probe["norm"].isin(name_only_dupes[name_only_dupes > 1].index)]

spread_name_only = (
    name_only_dupe_groups.groupby("norm", group_keys=False)
    .apply(max_pairwise_km, include_groups=False)
    .rename("max_pairwise_km")
)

print(f"Name-only duplicate groups: {len(spread_name_only)}")
print(f"Max spread:  {spread_name_only.max():.2f} km")
print(f"Mean spread: {spread_name_only.mean():.2f} km")

## Tier 1: Unambiguous Names — Join on `norm` Alone

In [ ]:
ambiguous_norms = set(name_only_dupes[name_only_dupes > 1].index)

unambiguous_gov = gov[~gov["norm"].isin(ambiguous_norms)]
ambiguous_gov = gov[gov["norm"].isin(ambiguous_norms)]

print(f"Unambiguous records: {len(unambiguous_gov)}")
print(f"Ambiguous records: {len(ambiguous_gov)}")

unambiguous_merged = unambiguous_gov.merge(geo, on="norm", how="left", indicator=True)

assert not unambiguous_merged["norm"].duplicated().any(), "❌ Unexpected duplicate match"
print(f"✅ Unambiguous settlements matched: {len(unambiguous_merged)}")

## Tier 2: Ambiguous Names — Join on (`norm`, `county_code`)

GeoNames' admin1_code corresponds to county codes.

But GeoNames' admin1_code numbering doesn't match KSH's county_code numbering — crosswalked by name via GeoNames' published admin1CodesASCII.txt.

### Step 1 — `county_code` Mapping Check

In [ ]:
GEONAMES_ADMIN1_TO_KSH_COUNTY = {
    1: 3, 2: 2, 3: 4, 4: 5, 5: 1, 6: 6, 8: 7, 9: 8, 10: 9, 11: 10,
    12: 11, 14: 12, 16: 13, 17: 14, 18: 15, 20: 16, 21: 17, 22: 18,
    23: 19, 24: 20,
}

ambiguous_geo = geo[geo["norm"].isin(ambiguous_norms)].copy()
ambiguous_geo["county_code"] = (
    ambiguous_geo["admin1 code"]
    .map(GEONAMES_ADMIN1_TO_KSH_COUNTY)
)

unresolved_norms = (
    ambiguous_geo.groupby("norm")["county_code"]
    .apply(lambda s: s.notna().any())
)
bad_norms = unresolved_norms[~unresolved_norms].index.tolist()
assert not bad_norms, (
    f"❌ {len(bad_norms)} norms have no county_code mapped: "
    f"{bad_norms}"
)
print("✅ All norms got county_code mapped => join is possible based on (name, county_code)")


### Step 2 — Perform Join on (`norm`, `county_code`)

In [ ]:
ambiguous_merged = ambiguous_gov.merge(
    ambiguous_geo, on=["norm", "county_code"], how="left", indicator=True
)

unmatched = ambiguous_merged[ambiguous_merged["_merge"] != "both"]
assert unmatched.empty, (
    f"❌ {len(unmatched)} norms have no match: "
    f"{unmatched}"
)
print("✅ All KSH settlements successfully matched to GeoNames")

## Tier 3: Resolve Residual Duplicates Manually

A handful of names still match more than one GeoNames record within
the *same* county — a GeoNames data redundancy, not a real-world
ambiguity. Resolved by preferring higher population, then more recent
modification date as a weaker fallback.

### Step 1 — Find Remaining Duplicates

In [ ]:
name_county_dupes = ambiguous_merged.groupby("norm").size()
dupe_mask = ambiguous_merged["norm"].isin(name_county_dupes[name_county_dupes > 1].index)
name_county_dupe_groups = ambiguous_merged[dupe_mask]

spread_name_county = (
    name_county_dupe_groups.groupby("norm", group_keys=False)
    .apply(max_pairwise_km, include_groups=False)
    .rename("max_pairwise_km")
)

display(spread_name_county)


### Step 2 — Resolve Duplicates

In [ ]:
ambiguous_merged_resolved = (
    ambiguous_merged
    .sort_values(["population", "modification date"], ascending=False, kind="stable")
    .drop_duplicates(subset="ksh_code", keep="first")
)

assert ambiguous_merged_resolved["ksh_code"].duplicated().sum() == 0, (
    "❌ Still have duplicate KSH settlements after resolution"
)
print(f"✅ Resolved {len(ambiguous_merged) - len(ambiguous_merged_resolved)} duplicate rows")

## Core Assertion

In [ ]:
merged = pd.concat([ambiguous_merged_resolved, unambiguous_merged], ignore_index=True)

assert merged["ksh_code"].duplicated().sum() == 0, (
    "❌ Some KSH settlement appears more than once in the final result"
)

total = len(gov)
matched = len(merged[merged["_merge"] == "both"])

print(f"Total KSH settlements: {total}")
print(f"Matched settlements: {matched}")
print(f"Coverage: {matched / total:.2%}")

assert matched == total, f"❌ Coverage incomplete: {matched}/{total}"
print("✅ All KSH settlements successfully matched to GeoNames")